# Shift-Scan: Results Analysis & Ablation Notebook (No Checkpoint Required)

This notebook analyzes **existing Shift-Scan experiment outputs** without loading or requiring a `.pt` model checkpoint.

It supports:
- Uploading existing JSON, CSV, ZIP, and log outputs
- Extracting experiment result files from ZIP archives
- Summarizing recorded metrics for variants such as `nearest`, `harpq`, and `dyadic2`
- Comparing reported perplexity, accuracy, and calibration values where available
- Plotting metrics already present in the outputs
- Exporting a consolidated results table

**This is results analysis, not fresh model evaluation or inference.** Any metrics shown are reported in the uploaded files; this notebook does not generate new model predictions. Missing metrics remain missing rather than being estimated.

## 1. Runtime setup

A GPU is not required for this results-analysis notebook. A standard Colab CPU runtime is sufficient.

In [ ]:
import os, json, zipfile, csv, glob
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/shift_scan_results")
ROOT.mkdir(parents=True, exist_ok=True)
print("Results directory:", ROOT)

## 2. Upload your previous experiment outputs

Upload your existing output ZIP archives and/or JSON/CSV files (for example, calibration JSON, seed summaries, evaluation JSON, CSV comparisons, or logs). You do **not** need to upload a `.pt` file.

If files are already present in Google Drive, you may instead mount Drive and copy the results folder into `ROOT`.

In [ ]:
from google.colab import files
uploaded = files.upload()

for name, content in uploaded.items():
    dest = ROOT / name
    dest.write_bytes(content)
    print("Saved:", dest)

### Optional: use files from Google Drive

Run this cell only if your files are stored in Drive. Change `DRIVE_RESULTS_DIR` to the folder containing your experiment outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT this folder path to your own Drive location:
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/Shift-Scan-Results")

if DRIVE_RESULTS_DIR.exists():
    import shutil
    for item in DRIVE_RESULTS_DIR.iterdir():
        target = ROOT / item.name
        if item.is_file():
            shutil.copy2(item, target)
        elif item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
    print("Copied Drive outputs into", ROOT)
else:
    print("Folder not found. Edit DRIVE_RESULTS_DIR and rerun this cell.")

## 3. Extract ZIP archives

This extracts uploaded archives into separate subfolders and skips unsafe paths outside the results directory.

In [ ]:
for zip_path in ROOT.rglob("*.zip"):
    extract_dir = zip_path.parent / (zip_path.stem + "_extracted")
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = (extract_dir / member.filename).resolve()
            if extract_dir.resolve() not in target.parents and target != extract_dir.resolve():
                print("Skipped unsafe archive path:", member.filename)
                continue
            zf.extract(member, extract_dir)
    print("Extracted:", zip_path.name, "->", extract_dir)

## 4. Inventory available result files

In [ ]:
json_files = sorted(ROOT.rglob("*.json"))
csv_files = sorted(ROOT.rglob("*.csv"))
log_files = sorted(list(ROOT.rglob("*.log")) + list(ROOT.rglob("*.txt")))

print(f"JSON files: {len(json_files)}")
for p in json_files: print(" -", p.relative_to(ROOT))
print(f"\\nCSV files: {len(csv_files)}")
for p in csv_files: print(" -", p.relative_to(ROOT))
print(f"\\nLog/text files: {len(log_files)}")
for p in log_files: print(" -", p.relative_to(ROOT))

## 5. Read JSON and CSV files

The loader keeps each file's source filename attached so you can trace reported metrics back to the output.

In [ ]:
json_data = {}
for p in json_files:
    try:
        with open(p, "r", encoding="utf-8") as f:
            json_data[str(p.relative_to(ROOT))] = json.load(f)
    except Exception as e:
        print("Could not parse JSON:", p.name, "|", e)

csv_data = {}
for p in csv_files:
    try:
        csv_data[str(p.relative_to(ROOT))] = pd.read_csv(p)
    except Exception as e:
        print("Could not parse CSV:", p.name, "|", e)

print("Loaded JSON:", len(json_data), "| CSV:", len(csv_data))

## 6. Show structured JSON summaries

This displays the parsed JSON objects. Use this to inspect which metrics are actually available in each output.

In [ ]:
for filename, obj in json_data.items():
    print("\\n" + "="*90)
    print(filename)
    print("="*90)
    if isinstance(obj, dict):
        print(json.dumps(obj, indent=2)[:12000])
    else:
        print(str(obj)[:12000])

## 7. Build a comparison table from seed/variant summaries

This parser handles the known nested summary pattern (`runs` → variant → `train_eval_step_*` and `downstream`) as well as simple JSON dictionaries. Unrecognized files are left out of this structured table and remain available in the inventory above.

In [ ]:
rows = []

def add_summary_rows(filename, obj):
    if not isinstance(obj, dict):
        return
    runs = obj.get("runs")
    if isinstance(runs, dict):
        for run_name, run in runs.items():
            if not isinstance(run, dict): continue
            variant = run.get("variant", run_name)
            train_eval = {}
            for k, v in run.items():
                if isinstance(k, str) and k.startswith("train_eval_step_") and isinstance(v, dict):
                    train_eval = v
                    break
            downstream = run.get("downstream", {})
            lambada = downstream.get("lambada_openai", {}) if isinstance(downstream, dict) else {}
            wiki = downstream.get("wikitext", {}) if isinstance(downstream, dict) else {}
            rows.append({
                "source_file": filename,
                "run": run_name,
                "variant": variant,
                "student_ppl": train_eval.get("student_ppl"),
                "s_ce": train_eval.get("s_ce"),
                "state_mse": train_eval.get("state_mse"),
                "lambada_accuracy": lambada.get("acc"),
                "lambada_perplexity": lambada.get("perplexity"),
                "wikitext_word_perplexity": wiki.get("word_perplexity"),
                "wikitext_byte_perplexity": wiki.get("byte_perplexity"),
                "bits_per_byte": wiki.get("bits_per_byte"),
            })

for filename, obj in json_data.items():
    add_summary_rows(filename, obj)

results_df = pd.DataFrame(rows)
if len(results_df):
    display(results_df)
else:
    print("No recognized nested run-summary JSON found. Inspect JSON files in Section 6 and/or load CSV tables directly.")

## 8. Compare reported metrics visually

Charts are created only for columns with numeric values in the parsed summary table. They show recorded values, not fresh evaluation.

In [ ]:
if not results_df.empty:
    plot_cols = [
        "student_ppl", "s_ce", "state_mse", "lambada_accuracy",
        "lambada_perplexity", "wikitext_word_perplexity",
        "wikitext_byte_perplexity", "bits_per_byte"
    ]
    for metric in plot_cols:
        vals = pd.to_numeric(results_df[metric], errors="coerce")
        plot_df = results_df.loc[vals.notna(), ["variant", metric]].copy()
        if plot_df.empty:
            continue
        plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")
        plt.figure(figsize=(8, 4))
        plt.bar(plot_df["variant"].astype(str), plot_df[metric])
        plt.title(f"Recorded {metric} by variant")
        plt.xlabel("Variant")
        plt.ylabel(metric)
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.show()
else:
    print("No structured results to plot yet.")

## 9. Inspect CSV comparison tables

In [ ]:
for filename, df in csv_data.items():
    print("\\n###", filename, "| shape:", df.shape)
    display(df.head(30))

## 10. Export consolidated results

Export the structured comparison table, if one was created. The CSV will contain only metrics found in the input JSON summaries.

In [ ]:
if not results_df.empty:
    export_path = ROOT / "shift_scan_consolidated_results.csv"
    results_df.to_csv(export_path, index=False)
    print("Saved:", export_path)
    files.download(str(export_path))
else:
    print("No structured summary table available to export. You can still export or inspect the original CSVs.")

## Interpretation notes

- Values in this notebook are imported from the provided files.
- Missing values are left blank/NaN; they are not guessed or filled.
- A training log or metric JSON is evidence of reported experiment results, but is not itself a model checkpoint.
- To calculate new perplexity, run fresh inference, or resume training, the actual model weights are still required.
- For a fair ablation, compare variants only when their dataset, token budget, sequence length, seed, and evaluation protocol match.